# Spatial Metrics and Backgrounds

This notebook covers the metrics that govern background noise contributions:
**core area** and **occulter transmission**. The physical size of the PSF
core dictates how much background noise enters the photometric aperture.

## Theory

### Core Area

The core area $\Omega$, measured in $(\lambda/D)^2$, represents the effective
solid angle of the PSF core. It enters the ETC in two ways:

1. **Detector pixels**: $N_{\rm pix} = \Omega / \theta_{\rm det}^2$ gives
   the number of detector pixels subtended by the core
2. **Background scaling**: All extended-source backgrounds (zodiacal dust,
   exozodiacal dust, detector noise) scale with $\Omega$

For a **fixed circular aperture**, the core area is constant:
$\Omega = N_{ap} \Delta\theta^2$, the fractionally weighted area equal to $\pi r_{ap}^2$. For the **PSF truncation ratio** method, the mask
shape adapts to the PSF at each separation, so $\Omega$ varies.

### Occulter Transmission

Coronagraphs also reject light from extended background sources such as
zodiacal and exozodiacal dust. The occulter transmission is the radial
profile of the `sky_trans.fits` mask. In the ETC, it directly scales the
background count rates $C_{bz}$ and $C_{bez}$.

**API**: Use `coro.occulter_transmission(separation)` for the
radial profile, or access the raw 2D map via `coro.sky_trans.data`.

```python
seps = coro.curves().separation_lod
occ_trans = coro.occulter_transmission(seps)
# Or access the 2D sky transmission map:
sky_map = coro.sky_trans.data  # 2D array
```

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import hwostyle
from yippy import viz

hwostyle.use("light")
import numpy as np
from matplotlib import animation
from IPython.display import HTML
from yippy import fetch_yip
from yippy import Coronagraph
from optixstuff.apertures import Circular, HalfMaximum
from optixstuff.performance import (
    PEAK_SEARCH_RADIUS_LOD,
    _window_peak,
    _refine_peak,
    _level_set_weights,
    CORE_OVERSAMPLE,
    core_subpixel_window,
    core_window_half_width_pix,
    truncation_core,
)

import logging; logging.getLogger("yippy").setLevel(logging.ERROR)

yip_path = fetch_yip("eac1_aavc_2d")
coro = Coronagraph(yip_path, rules=(Circular(0.7), HalfMaximum(0.3), HalfMaximum(0.5)))

from optixstuff.apertures import DEFAULT_APERTURE_RULE
from yippy.performance import stellar_intensity_curves

optics = coro
backend = coro
rule = DEFAULT_APERTURE_RULE
curves = optics.curves(rule)
print(f"Coronagraph: {coro.yip_path.name} (Amplitude Apodized Vortex Coronagraph, generated by Susan Redmond)")
print(f"Pixel scale: {coro.grid.step}")
print(f"IWA: {curves.iwa_lod:.2f}, Data extent: {curves.data_extent_lod:.2f}")

---
## Core Area: Fixed Aperture

For a circle fully inside the image, the counted area is constant:
$\Omega = N_{ap} \Delta\theta^2$ in $(\lambda/D)^2$, approaching $\pi r_{ap}^2$ in the continuum limit. At the image edge it is clipped.

An alternative is to fit a 2D Gaussian to each PSF and compute
$\Omega = \pi \cdot \text{FWHM}_x \cdot \text{FWHM}_y / 4$
(shown below for comparison, but not typically used in ETCs).

**API**: Read `coro.curves(rule).core_area_lod2` for fixed
apertures or truncation masks. The Gaussian comparison below fits the PSFs directly.

```python
from optixstuff.apertures import Circular, HalfMaximum
fixed = coro.curves(Circular(0.7))
seps, areas = fixed.separation_lod, fixed.core_area_lod2
truncated = coro.curves(HalfMaximum(0.3))
seps, areas = truncated.separation_lod, truncated.core_area_lod2
```

In [ ]:
sep_a, area_fixed = (coro.curves(Circular(0.7)).separation_lod, coro.curves(Circular(0.7)).core_area_lod2)
from astropy.modeling import fitting, models

# Fit each native PSF directly for this illustrative Gaussian comparison.
sep_g = sep_a
area_gauss = []
for separation in sep_g:
    psf = np.asarray(coro.source_psfs(None, separation, 0.0, grid=None).data[0])
    yy, xx = np.indices(psf.shape)
    py, px = np.unravel_index(np.argmax(psf), psf.shape)
    window = (xx - px)**2 + (yy - py)**2 <= (2.0 / coro.grid.step)**2
    initial = models.Gaussian2D(psf.max(), px, py, 0.5 / coro.grid.step, 0.5 / coro.grid.step)
    fitted = fitting.TRFLSQFitter()(initial, xx[window], yy[window], psf[window])
    # FWHM_x * FWHM_y = 8 ln(2) sigma_x sigma_y.
    area_gauss.append(2 * np.pi * np.log(2) * abs(fitted.x_stddev.value * fitted.y_stddev.value) * coro.grid.pixel_area)
area_gauss = np.asarray(area_gauss)

# Skip the first point where the Gaussian fit is unreliable
# (PSF at sep~0 is heavily suppressed, producing a bad fit)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sep_a[1:], area_fixed[1:], 's-', ms=5, color='#9C27B0',
        label='Fixed aperture (0.7 $\\lambda/D$)')
ax.plot(sep_g[1:], area_gauss[1:], 'o-', ms=5, color='#2196F3',
        label='Gaussian fit')
ax.set_xlabel('Separation [$\\lambda/D$]')
ax.set_ylabel('Core Area [$(\\lambda/D)^2$]')
ax.set_title(f'{coro.yip_path.name} -- Core Area (Fixed Aperture vs Gaussian Fit)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Core Area: Truncation Ratio Animation

With the PSF truncation ratio method, the core area varies with separation
because the mask shape adapts to the PSF structure. The animation below shows
the truncation mask (cyan contour) and the corresponding core area at each
position, compared with the constant fixed-aperture value.

yippy measures both truncation masks and circular apertures on the PSF's band-limited interpolant, ten subpixels per pixel side ({func}`~optixstuff.performance.truncation_core`, {func}`~optixstuff.performance.circular_aperture`). Curves use the package's own non-negative off-axis offsets by default and interpolate those measurements. Use {func}`~optixstuff.performance.performance_curves` with explicit `separations_lod` to measure a 0.25 $\lambda/D$ grid of synthesized PSFs as well. A YIP alone cannot validate the blended PSFs between its offsets. Sampling coarser than 0.5 $\lambda/D$ per pixel emits one warning per coronagraph. Fractional circle-square overlaps and tangent-plane level-set weights describe the aperture boundary. `Circular(radius_lod, centre="source")` defaults to the supplied source position without a peak search, keeping one aperture fixed in angle independent of pixelization. Select `centre="peak"` to follow the refined local PSF peak. Half-maximum apertures follow the peak by construction; their search stays within 1 $\lambda/D$ of each in-image nominal source. The aperture animations show in-image sources; public curves also retain native offsets outside the image.

In [ ]:
aperture_rule = HalfMaximum(0.5)
fraction = aperture_rule.fraction
pix_lod = coro.grid.step
half_width = core_window_half_width_pix(pix_lod)

from pathlib import Path
from types import SimpleNamespace
import astropy.io.fits as fits

# Read the shipped x-axis PSFs used to prepare the performance records.
offsets = np.asarray(fits.getdata(Path(yip_path) / "offax_psf_offset_list.fits"), dtype=jnp.result_type(float))
native_psfs = np.asarray(fits.getdata(Path(yip_path) / "offax_psf.fits"), dtype=jnp.result_type(float))
positions = []
with jax.enable_x64():
    for (x, y), psf in zip(offsets, native_psfs, strict=True):
        if x >= 0 and y == np.min(np.abs(offsets[:, 1])):
            px, py = coro.grid.unit_to_pixel(x, y)
            positions.append(SimpleNamespace(separation=float(x), psf=psf, px=float(px), py=float(py)))
positions = [p for p in positions if 0 <= p.px < coro.grid.shape[1] and 0 <= p.py < coro.grid.shape[0]]

area_frames = []
with jax.enable_x64():
    for pos in positions:
        from hwoutils.grids import Image
        from optixstuff.performance import _rule_samples

        image = Image(pos.psf, coro.grid, (), "per_pixel", "stellar_flux_fraction")
        source_position = coro.grid.pixel_to_unit(pos.px, pos.py)
        fine, weights, sy, sx, factor = _rule_samples(
            aperture_rule, image, source_position, CORE_OVERSAMPLE
        )
        psf_os, mask = np.asarray(fine), np.asarray(weights)
        tp = float(np.sum(psf_os * mask) / factor**2)
        area_lod2 = float(mask.sum() * coro.grid.pixel_area / factor**2)
        area_frames.append({
            'sep': pos.separation,
            'full_psf': pos.psf,
            'psf_os': psf_os,
            'mask': mask,
            'area': area_lod2,
        })

# Compute global colorscale across all frames
global_peak_full = max(
    np.log10(np.maximum(f['full_psf'], 1e-20)).max() for f in area_frames
)
global_peak_mask = max(
    np.log10(np.maximum(f['psf_os'], 1e-20)).max() for f in area_frames
)

fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.2], wspace=0.3)
ax_full = fig.add_subplot(gs[0, 0])
ax_mask = fig.add_subplot(gs[0, 1])
ax_curve = fig.add_subplot(gs[0, 2])

# Full-frame PSF
f0 = area_frames[0]
log_full = np.log10(np.maximum(f0['full_psf'], 1e-20))

im_full = ax_full.imshow(log_full, origin='lower', cmap='magma',
                         vmin=global_peak_full - 5, vmax=global_peak_full)
ax_full.set_title('Off-axis PSF (full frame)')
ax_full.set_xlabel('x [pix]')
ax_full.set_ylabel('y [pix]')
ax_full.set_aspect('equal')

# Cropped oversampled PSF with mask
log_psf = np.log10(np.maximum(f0['psf_os'], 1e-20))

im = ax_mask.imshow(log_psf, origin='lower', cmap='magma',
                    vmin=global_peak_mask - 4, vmax=global_peak_mask)
contour_set = ax_mask.contour(f0['mask'].astype(float), levels=[0.5],
                             colors='cyan', linewidths=2)
ax_mask.set_title('Truncation mask (zoomed)')
ax_mask.set_xlabel('x [subpixel]')
ax_mask.set_aspect('equal')

title_fig = fig.suptitle('', fontsize=12)

# Core area curve
a_seps = [f['sep'] for f in area_frames]
a_vals = [f['area'] for f in area_frames]
np.testing.assert_allclose(a_vals, coro.core_area(np.asarray(a_seps), rule=aperture_rule), rtol=1e-6)
ax_curve.plot(a_seps, a_vals, 'o-', ms=4, color='#CCCCCC', alpha=0.3, zorder=1)
scatter_a = ax_curve.scatter([], [], s=80, color='#9C27B0', zorder=3)
line_a, = ax_curve.plot([], [], 'o-', ms=5, color='#9C27B0', zorder=2)
ax_curve.axvline(curves.iwa_lod, ls='--', color='gray', alpha=0.7,
                 label=f'IWA = {curves.iwa_lod:.1f}')
ax_curve.set_xlabel('Separation [$\\lambda/D$]')
ax_curve.set_ylabel('Core Area [$(\\lambda/D)^2$]')
ax_curve.set_title(f'Core Area (ratio = {fraction})')
ax_curve.legend()
ax_curve.grid(True, alpha=0.3)

def update_area(i):
    f = area_frames[i]

    # Update full-frame
    log_f = np.log10(np.maximum(f['full_psf'], 1e-20))
    im_full.set_data(log_f)

    # Update cropped mask view
    log_img = np.log10(np.maximum(f['psf_os'], 1e-20))
    im.set_data(log_img)
    im.set_extent([-0.5, log_img.shape[1]-0.5, -0.5, log_img.shape[0]-0.5])
    ax_mask.set_xlim(-0.5, log_img.shape[1]-0.5)
    ax_mask.set_ylim(-0.5, log_img.shape[0]-0.5)

    for coll in ax_mask.collections[:]:
        coll.remove()
    ax_mask.contour(f['mask'].astype(float), levels=[0.5],
                   colors='cyan', linewidths=2)

    title_fig.set_text(
        f"Sep = {f['sep']:.2f} $\\lambda/D$  |  "
        f"Core Area = {f['area']:.3f} $(\\lambda/D)^2$"
    )

    line_a.set_data(a_seps[:i+1], a_vals[:i+1])
    scatter_a.set_offsets([[f['sep'], f['area']]])
    return im_full, im, title_fig, line_a, scatter_a

anim = animation.FuncAnimation(fig, update_area, frames=len(area_frames),
                               interval=400, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

### Fixed vs Truncation Comparison

In [ ]:
sep_f, tp_fixed = (coro.curves(Circular(0.7)).separation_lod, coro.curves(Circular(0.7)).throughput)
sep_af, area_f = (coro.curves(Circular(0.7)).separation_lod, coro.curves(Circular(0.7)).core_area_lod2)

sep_tr, tp_trunc = (coro.curves(HalfMaximum(0.3)).separation_lod, coro.curves(HalfMaximum(0.3)).throughput)
sep_at, area_t = (coro.curves(HalfMaximum(0.3)).separation_lod, coro.curves(HalfMaximum(0.3)).core_area_lod2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sep_f, tp_fixed, 'o-', ms=4, color='#4CAF50',
         label='Fixed aperture (0.7 $\\lambda/D$)')
ax1.plot(sep_tr, tp_trunc, 's-', ms=4, color='#FF5722',
         label='Truncation ratio (0.3)')
ax1.set_xlabel('Separation [$\\lambda/D$]')
ax1.set_ylabel('Throughput')
ax1.set_title('Throughput')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(sep_af, area_f, 'o-', ms=4, color='#9C27B0',
         label='Fixed aperture (0.7 $\\lambda/D$)')
ax2.plot(sep_at, area_t, 's-', ms=4, color='#FF5722',
         label='Truncation ratio (0.3)')
ax2.set_xlabel('Separation [$\\lambda/D$]')
ax2.set_ylabel('Core Area [$(\\lambda/D)^2$]')
ax2.set_title('Core Area')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle(f'{coro.yip_path.name} -- Fixed Aperture vs PSF Truncation Ratio',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Note how the truncation-ratio core area **varies with separation** while
the fixed aperture is constant. When calculating exposure times, AYO loops
over multiple truncation ratios and picks the one that minimizes integration
time at each separation.

For a deeper comparison of these two aperture methods, including noise
metrics and AYO's optimization strategy, see the
[Aperture Methods Comparison](05_Aperture_Methods_Comparison.ipynb).


---
## Occulter Transmission

Occulter transmission represents how much light from spatially extended
background sources (zodiacal dust, exozodiacal dust) survives the coronagraph
mask. It is the radial profile of the `sky_trans.fits` mask.

In the ETC, it scales all extended-source backgrounds ($C_{bz}$, $C_{bez}$).

In [ ]:
from hwoutils.radial import radial_profile

data_extent_lod = curves.data_extent_lod
seps_ot, profile_ot, _ = radial_profile(coro.sky_trans)
seps_ot = np.asarray(seps_ot)
profile_ot = np.asarray(profile_ot)

# Only show up to the data extent
mask_extent = seps_ot <= data_extent_lod * 1.1
step = max(1, len(seps_ot[mask_extent]) // 25)
frame_indices = list(range(0, len(seps_ot[mask_extent]), step))

fig_ot = plt.figure(figsize=(14, 5))
gs_ot = fig_ot.add_gridspec(1, 2, width_ratios=[1, 1.3], wspace=0.3)
ax_sky = fig_ot.add_subplot(gs_ot[0, 0])
ax_ot_prof = fig_ot.add_subplot(gs_ot[0, 1])

# Sky transmission image
im_sky = ax_sky.imshow(np.asarray(coro.sky_trans.data), origin='lower', cmap='magma',
                       extent=coro.sky_trans.grid.extent(), vmin=0, vmax=1)
ax_sky.set_aspect('equal')
annulus_in = plt.Circle((0, 0), 0, fill=False,
                        ec='white', lw=2)
annulus_out = plt.Circle((0, 0), 0, fill=False,
                         ec='white', lw=2)
ax_sky.add_patch(annulus_in)
ax_sky.add_patch(annulus_out)
ax_sky.set_title('Sky Transmission Map')
ax_sky.set_xlabel('x [$\\lambda/D$]')
ax_sky.set_ylabel('y [$\\lambda/D$]')
plt.colorbar(im_sky, ax=ax_sky, shrink=0.8)

# Profile plot
ax_ot_prof.plot(seps_ot[mask_extent], profile_ot[mask_extent], '-',
                color='#CCCCCC', alpha=0.3, lw=2, zorder=1)
line_ot, = ax_ot_prof.plot([], [], '-', color='#4CAF50', lw=2, zorder=2)
dot_ot = ax_ot_prof.scatter([], [], s=80, color='#4CAF50', zorder=3)
ax_ot_prof.axvline(curves.iwa_lod, ls='--', color='gray', alpha=0.7,
                   label=f'IWA = {curves.iwa_lod:.1f}')
ax_ot_prof.axvline(data_extent_lod, ls='--', color='gray', alpha=0.5,
                   label=f'Data extent = {data_extent_lod:.0f}')
ax_ot_prof.set_xlabel('Separation [$\\lambda/D$]')
ax_ot_prof.set_ylabel('Occulter Transmission')
ax_ot_prof.set_title('Radial Profile')
ax_ot_prof.set_ylim(-0.05, 1.05)
ax_ot_prof.set_xlim(0, data_extent_lod * 1.1)
ax_ot_prof.legend()
ax_ot_prof.grid(True, alpha=0.3)

title_ot = fig_ot.suptitle('', fontsize=12)

def update_ot(frame_num):
    idx = frame_indices[frame_num]
    r_lod = seps_ot[idx]
    half_width = (seps_ot[1] - seps_ot[0]) / 2

    annulus_in.set_radius(max(0, r_lod - half_width))
    annulus_out.set_radius(r_lod + half_width)

    line_ot.set_data(seps_ot[:idx+1], profile_ot[:idx+1])
    dot_ot.set_offsets([[r_lod, profile_ot[idx]]])

    title_ot.set_text(
        f'Sep = {r_lod:.1f} $\\lambda/D$  |  '
        f'Transmission = {profile_ot[idx]:.3f}'
    )
    return annulus_in, annulus_out, line_ot, dot_ot, title_ot

anim_ot = animation.FuncAnimation(fig_ot, update_ot,
                                  frames=len(frame_indices),
                                  interval=200, blit=False)
plt.close(fig_ot)
HTML(anim_ot.to_jshtml())

In [ ]:
sep_ot, occ_trans = (curves.separation_lod, optics.occulter_transmission(curves.separation_lod))
result = viz.plot_performance_curve(
    sep_ot, occ_trans, metric="occulter_transmission"
)
result.ax.set_title(f"{coro.yip_path.name} -- Occulter Transmission")
plt.show()